In [ ]:
!pip install numpy


In [ ]:
import numpy as np
import wave
import os
import json
from collections import deque

class VADConfig:
    SAMPLE_RATE       = 16000
    FRAME_DURATION_MS = 30
    ENERGY_THRESHOLD  = 0.02
    ZCR_THRESHOLD     = 0.15
    SPEECH_PAD_MS     = 300
    MIN_SPEECH_MS     = 250
    SMOOTHING_FRAMES  = 5

class VADEngine:
    def __init__(self, config=None):
        self.config = config or VADConfig()
        self.frame_size = int(self.config.SAMPLE_RATE * self.config.FRAME_DURATION_MS / 1000)
        self._decision_buffer = deque(maxlen=self.config.SMOOTHING_FRAMES)

    def _compute_energy(self, frame):
        if len(frame) == 0:
            return 0.0
        rms = np.sqrt(np.mean(frame.astype(np.float64) ** 2))
        return float(rms / 32768.0)

    def _compute_zcr(self, frame):
        if len(frame) < 2:
            return 0.0
        signs = np.sign(frame.astype(np.float32))
        crossings = np.sum(np.abs(np.diff(signs))) / 2.0
        return float(crossings / len(frame))

    def _compute_spectral_flatness(self, frame):
        if len(frame) < 2:
            return 1.0
        fft_mag = np.abs(np.fft.rfft(frame.astype(np.float32)))
        fft_mag = fft_mag + 1e-10
        geometric_mean = np.exp(np.mean(np.log(fft_mag)))
        arithmetic_mean = np.mean(fft_mag)
        if arithmetic_mean == 0:
            return 1.0
        return float(geometric_mean / arithmetic_mean)

    def _is_speech_frame(self, frame):
        energy   = self._compute_energy(frame)
        zcr      = self._compute_zcr(frame)
        flatness = self._compute_spectral_flatness(frame)
        energy_ok  = energy   > self.config.ENERGY_THRESHOLD
        zcr_ok     = zcr      < self.config.ZCR_THRESHOLD * 2
        noise_ok   = flatness < 0.85
        return energy_ok and zcr_ok and noise_ok

    def _smooth_decision(self, raw_decision):
        self._decision_buffer.append(1 if raw_decision else 0)
        avg = sum(self._decision_buffer) / len(self._decision_buffer)
        return avg > 0.5

    def _frames_to_segments(self, decisions, total_samples):
        sr = self.config.SAMPLE_RATE
        fs = self.frame_size
        pad_ms = self.config.SPEECH_PAD_MS
        min_ms = self.config.MIN_SPEECH_MS
        segments = []
        in_speech = False
        seg_start = 0
        pad_counter = 0
        for i, is_speech in enumerate(decisions):
            if is_speech:
                if not in_speech:
                    seg_start = max(0, i * fs - int(sr * pad_ms / 1000))
                    in_speech = True
                pad_counter = 0
            else:
                if in_speech:
                    pad_counter += 1
                    pad_frames = int(pad_ms / self.config.FRAME_DURATION_MS)
                    if pad_counter >= pad_frames:
                        seg_end = min(total_samples, i * fs + int(sr * pad_ms / 1000))
                        in_speech = False
                        pad_counter = 0
                        start_sec = seg_start / sr
                        end_sec = seg_end / sr
                        duration_ms = (end_sec - start_sec) * 1000
                        if duration_ms >= min_ms:
                            segments.append({"start_sec": round(start_sec, 3), "end_sec": round(end_sec, 3), "duration_ms": round(duration_ms, 1), "start_sample": seg_start, "end_sample": seg_end})
        if in_speech:
            seg_end = total_samples
            start_sec = seg_start / sr
            end_sec = seg_end / sr
            duration_ms = (end_sec - start_sec) * 1000
            if duration_ms >= min_ms:
                segments.append({"start_sec": round(start_sec, 3), "end_sec": round(end_sec, 3), "duration_ms": round(duration_ms, 1), "start_sample": seg_start, "end_sample": seg_end})
        return segments

    def process_samples(self, samples):
        self._decision_buffer.clear()
        decisions = []
        num_frames = len(samples) // self.frame_size
        for i in range(num_frames):
            frame = samples[i * self.frame_size : (i + 1) * self.frame_size]
            raw = self._is_speech_frame(frame)
            smooth = self._smooth_decision(raw)
            decisions.append(smooth)
        segments = self._frames_to_segments(decisions, len(samples))
        total_duration = len(samples) / self.config.SAMPLE_RATE
        speech_duration = sum(s["duration_ms"] for s in segments) / 1000
        return {"segments": segments, "num_speech_segments": len(segments), "total_duration_sec": round(total_duration, 3), "speech_duration_sec": round(speech_duration, 3), "silence_duration_sec": round(total_duration - speech_duration, 3), "speech_ratio": round(speech_duration / total_duration, 3) if total_duration > 0 else 0, "frame_decisions": decisions}

    def process_file(self, filepath):
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File nahi mili: {filepath}")
        with wave.open(filepath, 'rb') as wf:
            n_channels = wf.getnchannels()
            sampwidth  = wf.getsampwidth()
            framerate  = wf.getframerate()
            n_frames   = wf.getnframes()
            raw_data   = wf.readframes(n_frames)
        if sampwidth == 2:
            samples = np.frombuffer(raw_data, dtype=np.int16)
        else:
            samples = (np.frombuffer(raw_data, dtype=np.uint8).astype(np.int16) - 128) * 256
        if n_channels == 2:
            samples = samples[::2]
        result = self.process_samples(samples)
        result["file"] = filepath
        result["sample_rate"] = framerate
        return result

def generate_demo_audio(filepath="demo_audio.wav"):
    sr = 16000
    total = []
    def silence(d): return np.zeros(int(sr*d), dtype=np.int16)
    def speech_tone(d, freq=200):
        t = np.linspace(0, d, int(sr*d))
        return (np.sin(2*np.pi*freq*t)*8000).astype(np.int16) + (np.random.randn(int(sr*d))*200).astype(np.int16)
    total.append(silence(0.5))
    total.append(speech_tone(1.0, 150))
    total.append((np.random.randn(int(sr*0.3))*300).astype(np.int16))
    total.append(speech_tone(0.8, 180))
    total.append(silence(0.4))
    total.append(speech_tone(1.2, 160))
    total.append(silence(0.5))
    audio = np.concatenate(total)
    with wave.open(filepath, 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(audio.tobytes())
    return filepath # Added return statement

def print_result(result):
    print(json.dumps(result, indent=2))

In [ ]:
audio_file = generate_demo_audio("demo_audio.wav")
vad = VADEngine()
result = vad.process_file(audio_file)
print_result(result)

{
  "segments": [
    {
      "start_sec": 0.24,
      "end_sec": 2.13,
      "duration_ms": 1890.0,
      "start_sample": 3840,
      "end_sample": 34080
    },
    {
      "start_sec": 1.56,
      "end_sec": 3.24,
      "duration_ms": 1680.0,
      "start_sample": 24960,
      "end_sample": 51840
    },
    {
      "start_sec": 2.76,
      "end_sec": 4.7,
      "duration_ms": 1940.0,
      "start_sample": 44160,
      "end_sample": 75200
    }
  ],
  "num_speech_segments": 3,
  "total_duration_sec": 4.7,
  "speech_duration_sec": 5.51,
  "silence_duration_sec": -0.81,
  "speech_ratio": 1.172,
  "frame_decisions": [
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true,
    true

In [ ]:
import unittest

class TestVADEngine(unittest.TestCase):
    def setUp(self):
        self.config = VADConfig()
        self.vad = VADEngine(self.config)
        self.sr = self.config.SAMPLE_RATE
    def _make_silence(self, d): return np.zeros(int(self.sr*d), dtype=np.int16)
    def _make_speech(self, d, freq=200):
        t = np.linspace(0, d, int(self.sr*d))
        return (np.sin(2*np.pi*freq*t)*10000).astype(np.int16)
    def _make_breathing(self, d): return (np.random.randn(int(self.sr*d))*250).astype(np.int16)
    def test_1_silence(self):
        r = self.vad.process_samples(self._make_silence(2.0))
        self.assertEqual(r["num_speech_segments"], 0)
        print("✅ Test 1 passed: Silence detected")
    def test_2_speech(self):
        r = self.vad.process_samples(self._make_speech(1.5))
        self.assertGreater(r["num_speech_segments"], 0)
        print("✅ Test 2 passed: Speech detected")
    def test_3_segments(self):
        audio = np.concatenate([self._make_silence(0.3), self._make_speech(1.0), self._make_silence(0.5), self._make_speech(0.8), self._make_silence(0.3)])
        r = self.vad.process_samples(audio)
        self.assertGreaterEqual(r["num_speech_segments"], 1)
        print(f"✅ Test 3 passed: {r['num_speech_segments']} segments found")
    def test_4_energy(self):
        e_s = self.vad._compute_energy(self._make_silence(0.03))
        e_sp = self.vad._compute_energy(self._make_speech(0.03))
        self.assertGreater(e_sp, e_s)
        print(f"✅ Test 4 passed: Energy silence={e_s:.4f} speech={e_sp:.4f}")
    def test_5_zcr(self):
        z_sp = self.vad._compute_zcr(self._make_speech(0.03, 150))
        z_br = self.vad._compute_zcr(self._make_breathing(0.03))
        print(f"✅ Test 5 passed: ZCR speech={z_sp:.4f} breathing={z_br:.4f}")
    def test_6_short_noise(self):
        audio = np.concatenate([self._make_silence(0.5), self._make_speech(0.1), self._make_silence(0.5)])
        r = self.vad.process_samples(audio)
        for seg in r["segments"]: self.assertGreaterEqual(seg["duration_ms"], self.config.MIN_SPEECH_MS)
        print("✅ Test 6 passed: Short noise filtered")
    def test_7_file(self):
        fp = "/tmp/test.wav"
        audio = np.concatenate([self._make_silence(0.5), self._make_speech(1.0), self._make_silence(0.5)])
        with wave.open(fp,'wb') as wf: wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(self.sr); wf.writeframes(audio.tobytes())
        r = self.vad.process_file(fp)
        self.assertIn("segments", r)
        os.remove(fp)
        print("✅ Test 7 passed: File processing works")
    def test_8_fields(self):
        r = self.vad.process_samples(self._make_silence(1.0))
        for f in ["segments","num_speech_segments","total_duration_sec","speech_duration_sec"]: self.assertIn(f, r)
        print("✅ Test 8 passed: All fields present")

loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestVADEngine)
runner = unittest.TextTestRunner(verbosity=0)
result = runner.run(suite)
print(f"total {result.testsRun} tests PASS!" if result.wasSuccessful() else "failed")

----------------------------------------------------------------------
Ran 8 tests in 0.038s

OK


✅ Test 1 passed: Silence detected
✅ Test 2 passed: Speech detected
✅ Test 3 passed: 2 segments found
✅ Test 4 passed: Energy silence=0.0000 speech=0.2156
✅ Test 5 passed: ZCR speech=0.0188 breathing=0.4729
✅ Test 6 passed: Short noise filtered
✅ Test 7 passed: File processing works
✅ Test 8 passed: All fields present
total 8 tests PASS!
